# ⚙️ 03 — Baseline Modeling

This notebook builds and evaluates the `baseline prediction model` for Taiwan’s air quality dataset.

The purpose is to establish `a reference performance benchmark` before moving on to more advanced models.

## 🧠 01 — Notebook Metadata

Basic setup and context for baseline model training.

💡 Content:

- Import all necessary libraries (`pandas`, `sklearn`, `matplotlib`, `seaborn`)

- Load configuration paths from `config.py`

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sklearn

from src.config import PROCESSED_DIR
from src.features.feature_engineering import (
    add_rolling_features,
    clip_pollutants,
    handle_outliers_iqr,
    log_transform_features,
)
from src.modeling.train_baseline import load_cleaned_data
from src.utils.emoji_log import success

success("Modules imported successfully. Ready for baseline modeling!")

✅ Modules imported successfully. Ready for baseline modeling!


## 📂 02 — Load and Prepare Data

Load the processed and feature-engineered dataset (e.g., `Taiwan` or per-city data)
and prepare it for modeling.

💡 Content:

- Use `load_cleaned_data("Taiwan")`

- Apply `log_transform_features()` if not yet transformed

- Inspect shape, columns, and basic statistics (`df.shape`, `df.head()`, `df.describe()`)

In [ ]:
df = load_cleaned_data("Taiwan")

# clip the pollutants
df_clip = clip_pollutants(df.copy())

# Add the rolling columns
df_rolling = add_rolling_features(df_clip)

# Set iqr
df_iqr = handle_outliers_iqr(df_rolling)

# smooth the skewes
df_log = log_transform_features(df_iqr)

📂 Loading CSV: C:\Users\dinni\OneDrive\桌面\air_pollution\data\processed\Taiwan.csv
✅ Read CSV successfully! Shape: (5823862, 25)
✅ The pollutants limit has been set.
⚠️ Skipping nox due to NO + NO2 already exist.
✅ Rolling features added.
✅ IQR has been set.
✅ Pollutants skewes has been smoothed.


,date,sitename,county,aqi,status,so2,co,o3,o3_8hr,pm10,...,o3_rolling_3d,o3_rolling_7d,pm10_rolling_3d,pm10_rolling_7d,pm2.5_rolling_3d,pm2.5_rolling_7d,no2_rolling_3d,no2_rolling_7d,no_rolling_3d,no_rolling_7d
0,2024-08-31 23:00:00,Hukou,Hsinchu_County,62.0,Moderate,0.641854,0.157004,3.583519,40.2,2.944439,...,35.000000,35.000000,18.000000,18.000000,17.000000,17.000000,2.300000,2.300000,0.300000,0.300000
54,2024-08-31 23:00:00,Magong,Penghu_County,74.0,Moderate,0.916291,0.207014,4.082609,62.8,2.890372,...,58.300000,58.300000,17.000000,17.000000,19.000000,19.000000,1.000000,1.000000,1.700000,1.700000
62,2024-08-31 23:00:00,Pingtung (Liuqiu),Pingtung_County,43.0,Good,0.336472,0.165514,3.618993,42.2,3.091042,...,36.300000,36.300000,21.000000,21.000000,9.000000,9.000000,3.800000,3.800000,0.300000,0.300000
61,2024-08-31 23:00:00,Tainan (Madou),Tainan_City,41.0,Good,0.530628,0.231112,3.173878,38.5,2.944439,...,22.900000,22.900000,18.000000,18.000000,7.000000,7.000000,12.500000,12.500000,1.100000,1.100000
60,2024-08-31 23:00:00,Kaohsiung (Hunei),Kaohsiung_City,37.0,Good,0.916291,0.165514,3.555348,40.8,2.639057,...,34.000000,34.000000,13.000000,13.000000,5.000000,5.000000,4.500000,4.500000,1.300000,1.300000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5823810,2016-11-25 13:00:00,Xindian,New_Taipei_City,29.0,Good,0.832909,0.322083,3.465736,31.0,2.484907,...,32.333333,29.857143,8.333333,7.285714,13.666667,11.142857,10.700000,13.442857,3.666667,3.571429
5823809,2016-11-25 13:00:00,Wanli,New_Taipei_City,34.0,Good,0.788457,0.182322,3.761200,40.0,3.465736,...,42.000000,42.000000,35.333333,33.714286,3.666667,6.142857,3.100000,2.814286,1.700000,1.614286
5823808,2016-11-25 13:00:00,Xizhi,New_Taipei_City,23.0,Good,0.530628,0.231112,3.367296,27.0,3.091042,...,29.666667,29.142857,19.666667,19.285714,5.666667,6.428571,10.333333,10.714286,3.666667,3.242857
5823807,2016-11-25 13:00:00,Keelung,Keelung_City,30.0,Good,0.741937,0.198851,3.637586,35.0,2.708050,...,37.666667,37.571429,13.333333,15.571429,4.333333,5.285714,3.366667,3.400000,1.633333,1.571429


In [5]:
df_log.shape

(5823862, 38)

In [7]:
df_log.head()

,date,sitename,county,aqi,status,so2,co,o3,o3_8hr,pm10,...,o3_rolling_3d,o3_rolling_7d,pm10_rolling_3d,pm10_rolling_7d,pm2.5_rolling_3d,pm2.5_rolling_7d,no2_rolling_3d,no2_rolling_7d,no_rolling_3d,no_rolling_7d
0,2024-08-31 23:00:00,Hukou,Hsinchu_County,62.0,Moderate,0.641854,0.157004,3.583519,40.2,2.944439,...,35.0,35.0,18.0,18.0,17.0,17.0,2.3,2.3,0.3,0.3
54,2024-08-31 23:00:00,Magong,Penghu_County,74.0,Moderate,0.916291,0.207014,4.082609,62.8,2.890372,...,58.3,58.3,17.0,17.0,19.0,19.0,1.0,1.0,1.7,1.7
62,2024-08-31 23:00:00,Pingtung (Liuqiu),Pingtung_County,43.0,Good,0.336472,0.165514,3.618993,42.2,3.091042,...,36.3,36.3,21.0,21.0,9.0,9.0,3.8,3.8,0.3,0.3
61,2024-08-31 23:00:00,Tainan (Madou),Tainan_City,41.0,Good,0.530628,0.231112,3.173878,38.5,2.944439,...,22.9,22.9,18.0,18.0,7.0,7.0,12.5,12.5,1.1,1.1
60,2024-08-31 23:00:00,Kaohsiung (Hunei),Kaohsiung_City,37.0,Good,0.916291,0.165514,3.555348,40.8,2.639057,...,34.0,34.0,13.0,13.0,5.0,5.0,4.5,4.5,1.3,1.3


In [8]:
df_log.describe()

,date,aqi,so2,co,o3,o3_8hr,pm10,pm2.5,no2,no,...,o3_rolling_3d,o3_rolling_7d,pm10_rolling_3d,pm10_rolling_7d,pm2.5_rolling_3d,pm2.5_rolling_7d,no2_rolling_3d,no2_rolling_7d,no_rolling_3d,no_rolling_7d
count,5823862,5.823862e+06,5.823841e+06,5.823841e+06,5.823841e+06,5.823841e+06,5.823607e+06,5.823791e+06,5.823837e+06,5.823837e+06,...,5.823841e+06,5.823841e+06,5.823607e+06,5.823607e+06,5.823791e+06,5.823791e+06,5.823837e+06,5.823837e+06,5.823837e+06,5.823837e+06
mean,2020-11-26 06:17:31.644631808,5.423027e+01,9.805545e-01,2.697061e-01,3.224398e+00,3.013913e+01,3.341305e+00,2.630785e+00,2.261632e+00,9.606313e-01,...,3.020793e+01,3.020763e+01,3.436091e+01,3.436037e+01,1.681923e+01,1.681913e+01,1.124373e+01,1.124357e+01,3.456005e+00,3.455955e+00
min,2016-11-25 13:00:00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2019-02-09 20:00:00,3.200000e+01,6.931472e-01,1.739533e-01,2.833213e+00,1.800000e+01,2.944439e+00,2.197225e+00,1.808289e+00,5.877867e-01,...,1.646667e+01,1.771429e+01,1.833333e+01,1.871429e+01,8.333333e+00,8.571429e+00,5.300000e+00,5.600000e+00,8.021264e-01,9.000000e-01
50%,2021-02-02 11:00:00,4.700000e+01,9.932518e-01,2.468601e-01,3.367296e+00,2.860000e+01,3.401197e+00,2.708050e+00,2.312535e+00,8.754687e-01,...,2.816667e+01,2.849784e+01,2.866667e+01,2.885714e+01,1.366667e+01,1.371429e+01,9.217967e+00,9.471429e+00,1.466667e+00,1.571429e+00
75%,2022-09-12 13:00:00,7.000000e+01,1.252763e+00,3.435897e-01,3.751854e+00,4.000000e+01,3.828641e+00,3.135494e+00,2.772589e+00,1.308333e+00,...,4.133333e+01,4.061429e+01,4.475105e+01,4.485714e+01,2.233333e+01,2.218958e+01,1.500000e+01,1.488571e+01,2.833333e+00,2.957143e+00
max,2024-08-31 23:00:00,5.000000e+02,1.749200e+00,5.538851e-01,4.394449e+00,1.358000e+02,4.460144e+00,3.784190e+00,3.429137e+00,1.879465e+00,...,1.565582e+02,1.366571e+02,1.000000e+03,9.285714e+02,3.586667e+02,1.882810e+02,1.208833e+02,8.242857e+01,2.946667e+02,2.367143e+02
std,NaN,2.978873e+01,4.003616e-01,1.237822e-01,7.337454e-01,1.578343e+01,6.774754e-01,7.189771e-01,6.930268e-01,5.205350e-01,...,1.756278e+01,1.615450e+01,2.330810e+01,2.237542e+01,1.206548e+01,1.160489e+01,8.298150e+00,7.828619e+00,7.842026e+00,7.239352e+00


## 🧩 03 — Define Features and Target

Identify the `predictors (X)` and the `target (y)` variable for modeling.

💡 Content:

- Target variable: `aqi`

- Exclude non-numeric or categorical columns (`date`, `county`, `sitename`)

- Verify feature matrix and target vector shape

- Optionally scale features using `StandardScaler`

## ✂️ 04 — Train-Test Split

Divide the dataset into training and testing sets for model validation.

💡 Content:

- Use `train_test_split` (e.g., 80% train / 20% test)

- Print sample sizes

- Check that distributions of `aqi` in both sets are consistent

## 🧮 05 — Build Baseline Model

Train a simple `Linear Regression` as the baseline.

💡 Content:

- `from sklearn.linear_model import LinearRegression`

- Fit the model on training data

- Print regression coefficients

- Plot predicted vs actual values

## 📈 06 — Evaluate Performance

Compute and visualize model evaluation metrics.

💡 Content:

- Use:

    `from sklearn.metrics import mean_absolute_error, r2_score`


- Calculate:

    - Mean Absolute Error (MAE)

    - R² (Coefficient of Determination)

- Plot residuals (y_test - y_pred)

- Visualize prediction scatter plot (sns.regplot or plt.scatter)

## 💾 07 — Save Model and Metrics

Store the trained baseline model for future comparison.

💡 Content:

- Save using:

    ```
    import joblib
    joblib.dump(model, MODEL_DIR / "baseline_linear.pkl")
    ```

- Save metrics summary as `.json` or `.csv` for tracking progress.

## 📊 08 — Model Interpretation

Explore which features influence AQI the most.

💡 Content:

- Use regression coefficients or `feature_importances_` (for tree models)

- Plot bar chart of top predictors

- Discuss correlation vs model weight differences

## ✅ 09 — Summary and Next Steps

Conclude baseline results and outline improvement ideas.

💡 Content:
### ✅ Baseline Summary
- Model: Linear Regression  
- MAE: 8.21  
- R²: 0.78  
- Key Features: PM2.5, PM10, NO₂, O₃  

### 🚀 Next Steps
- Try non-linear models (RandomForest, XGBoost)
- Add weather-based features (wind speed, direction)
- Perform hyperparameter tuning
